# 示例策略11: 海龟交易法策略

本策略通过计算CZCE.FG801和SHFE.rb1801的ATR.唐奇安通道和MA线,并:
上穿唐奇安通道且短MA在长MA上方则开多仓,下穿唐奇安通道且短MA在长MA下方则开空仓

若有 多/空 仓位则分别:

价格 跌/涨 破唐奇安平仓通道 上/下 轨则全平仓位,否则

根据 跌/涨 破持仓均价 -/+ x(x=0.5,1,1.5,2)倍ATR把仓位

回测数据为:CZCE.FG801和SHFE.rb1801的1min数据

回测时间为:2017-09-15 09:15:00到2017-10-01 15:00:00


In [2]:
import os
import sys
sys.path.insert(0, os.path.abspath('../'))
# 导入qteasy模块
import qteasy as qt
import pandas as pd
import numpy as np
print(qt.__version__, qt.QT_DATA_SOURCE)

2.0.0 mysql://localhost@3306/ts_db


In [ ]:
from qteasy.tafuncs import atr as ta_atr
'''
本策略通过计算CZCE.FG801和SHFE.rb1801的ATR.唐奇安通道和MA线,并:
上穿唐奇安通道且短MA在长MA上方则开多仓,下穿唐奇安通道且短MA在长MA下方则开空仓
若有 多/空 仓位则分别:
价格 跌/涨 破唐奇安平仓通道 上/下 轨则全平仓位,否则
根据 跌/涨 破持仓均价 -/+ x(x=0.5,1,1.5,2)倍ATR把仓位
回测数据为:CZCE.FG801和SHFE.rb1801的1min数据
回测时间为:2017-09-15 09:15:00到2017-10-01 15:00:00
'''

class TurtleTrading(qt.GeneralStg):
    
    def __init__(self, pars: tuple = (55, 20, 10, 60, 20)):
        super().__init__(
                pars=pars,
                par_count=5,
                par_types=['int', 'int', 'int', 'int', 'int'],  # 唐奇安开仓通道.唐奇安平仓通道.短ma.长ma.ATR的参数
                par_range=[(10, 60), (10, 60), (10, 60), (10, 60), (10, 60)],
                name='TurtleTrading',
                description='根据ATR, 唐奇安通道以及MA线生成交易信号',
                strategy_run_timing='close',  # 在周期结束（收盘）时运行
                strategy_run_freq='1min',  # 每份钟执行一次调整
                strategy_data_types='high, low, close',  # 使用分钟三种k线价格生成策略
                data_freq='1min',  # 数据频率（包括股票数据和参考数据）
                window_length=100,
                use_latest_data_cycle=False,  # 高频数据不需要使用当前数据区间
                reference_data_types='',  # 不需要使用参考数据
        )
    
    def realize(self, h, r=None, t=None, pars=None):
        """策略输出PT信号，即仓位目标信号"""

        don_open, don_close, ma_short, ma_long, tar = self.get_pars('don_open', 'don_close', 'ma_short', 'ma_long', 'tar')

        # 读取最近N天的收盘价
        high, low, close = self.get_data('high_ANY_d', 'low_ANY_d', 'close_ANY_d')
        
        # 计算ATR
        atr = ta_atr(high, low, close, tar)
        
        # 计算唐奇安开仓和平仓通道
        upper_band = np.max(close[:, -don_open:])
        lower_band = np.max(close[:, -don_close:])
        
        # 计算开仓的资金比例
        percent = 0.8 / len(h)

        # 计算N天的平均价和标准差，并计算仓位阈值
        close_mean = np.nanmean(close, axis=1)
        close_std = np.nanstd(close, axis=1)
        hi_positive = close_mean + high_threshold * close_std
        low_positive = close_mean + low_threshold * close_std
        low_negative = close_mean - low_threshold * close_std
        hi_negative = close_mean - high_threshold * close_std

        # 根据当前的实际价格确定目标仓位，并将目标仓位作为信号输出
        pos = np.zeros_like(close_mean)
        pos = np.where(current_close > hi_positive, hi_pos, pos)
        pos = np.where(hi_positive >= current_close > low_positive, low_pos, pos)
        pos = np.where(low_positive >= current_close > low_negative, 0, pos)
        pos = np.where(low_negative >= current_close > hi_negative, - low_pos, pos)
        pos = np.where(current_close >= hi_negative, - hi_pos, pos)

        return pos
    
    
def init(context):
    # context.parameter分别为唐奇安开仓通道.唐奇安平仓通道.短ma.长ma.ATR的参数
    context.parameter = [55, 20, 10, 60, 20]
    context.tar = context.parameter[4]
    # context.goods交易的品种
    context.goods = ['CZCE.FG801', 'SHFE.rb1801']
    # context.ratio交易最大资金比率
    context.ratio = 0.8
    # 订阅context.goods里面的品种, bar频率为1min
    subscribe(symbols=context.goods, frequency='60s', count=101)
    # 止损的比例区间
def on_bar(context, bars):
    bar = bars[0]
    symbol = bar['symbol']
    recent_data = context.data(symbol=symbol, frequency='60s', count=101, fields='close,high,low')
    close = recent_data['close'].values[-1]
    # 计算ATR
    atr = talib.ATR(recent_data['high'].values, recent_data['low'].values, recent_data['close'].values,
                    timeperiod=context.tar)[-1]
    # 计算唐奇安开仓和平仓通道
    context.don_open = context.parameter[0] + 1
    upper_band = talib.MAX(recent_data['close'].values[:-1], timeperiod=context.don_open)[-1]
    context.don_close = context.parameter[1] + 1
    lower_band = talib.MIN(recent_data['close'].values[:-1], timeperiod=context.don_close)[-1]
    # 计算开仓的资金比例
    percent = context.ratio / float(len(context.goods))
    # 若没有仓位则开仓
    position_long = context.account().position(symbol=symbol, side=PositionSide_Long)
    position_short = context.account().position(symbol=symbol, side=PositionSide_Short)
    if not position_long and not position_short:
        # 计算长短ma线.DIF
        ma_short = talib.MA(recent_data['close'].values, timeperiod=(context.parameter[2] + 1))[-1]
        ma_long = talib.MA(recent_data['close'].values, timeperiod=(context.parameter[3] + 1))[-1]
        dif = ma_short - ma_long
        # 获取当前价格
        # 上穿唐奇安通道且短ma在长ma上方则开多仓
        if close > upper_band and (dif > 0):
            order_target_percent(symbol=symbol, percent=percent, order_type=OrderType_Market,
                                 position_side=PositionSide_Long)
            print(symbol, '市价单开多仓到比例: ', percent)
        # 下穿唐奇安通道且短ma在长ma下方则开空仓
        if close < lower_band and (dif < 0):
            order_target_percent(symbol=symbol, percent=percent, order_type=OrderType_Market,
                                 position_side=PositionSide_Short)
            print(symbol, '市价单开空仓到比例: ', percent)
    elif position_long:
        # 价格跌破唐奇安平仓通道全平仓位止损
        if close < lower_band:
            order_close_all()
            print(symbol, '市价单全平仓位')
        else:
            # 获取持仓均价
            vwap = position_long['vwap']
            # 获取持仓的资金
            money = position_long['cost']
            # 获取平仓的区间
            band = vwap - np.array([200, 2, 1.5, 1, 0.5, -100]) * atr
            grid_percent = float(pd.cut([close], band, labels=[0, 0.25, 0.5, 0.75, 1])[0]) * percent
            # 选择现有百分比和区间百分比中较小的值(避免开仓)
            target_percent = np.minimum(money / context.account().cash['nav'], grid_percent)
            if target_percent != 1.0:
                print(symbol, '市价单平多仓到比例: ', target_percent)
                order_target_percent(symbol=symbol, percent=target_percent, order_type=OrderType_Market,
                                     position_side=PositionSide_Long)
    elif position_short:
        # 价格涨破唐奇安平仓通道或价格涨破持仓均价加两倍ATR平空仓
        if close > upper_band:
            order_close_all()
            print(symbol, '市价单全平仓位')
        else:
            # 获取持仓均价
            vwap = position_short['vwap']
            # 获取持仓的资金
            money = position_short['cost']
            # 获取平仓的区间
            band = vwap + np.array([-100, 0.5, 1, 1.5, 2, 200]) * atr
            grid_percent = float(pd.cut([close], band, labels=[1, 0.75, 0.5, 0.25, 0])[0]) * percent
            # 选择现有百分比和区间百分比中较小的值(避免开仓)
            target_percent = np.minimum(money / context.account().cash['nav'], grid_percent)
            if target_percent != 1.0:
                order_target_percent(symbol=symbol, percent=target_percent, order_type=OrderType_Market,
                                     position_side=PositionSide_Short)
                print(symbol, '市价单平空仓到比例: ', target_percent)
